# 🔬 FakeInversion - Notebook 2: DDIM Inversion Feature Extraction

This notebook:
1. Loads Stable Diffusion v1.5 + BLIP models
2. Demonstrates inversion on sample images
3. Extracts 9-channel features for the full dataset
4. Saves features as .pt files for classifier training

In [ ]:
# Setup
import os, sys
PROJECT_DIR = '/content/fake_inversion'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from google.colab import drive
drive.mount('/content/drive')

import config
config.ensure_dirs()
config.print_config()

In [ ]:
# Demo: Visualize DDIM Inversion on a single image
from PIL import Image
from inversion.ddim_inverter import DDIMInverter
from evaluation.visualize import plot_inversion_features
import matplotlib.pyplot as plt

# Initialize inverter
inverter = DDIMInverter()
inverter.load_models()

# Pick a sample image
from utils import list_image_files

# Try a fake image
fake_dir = os.path.join(config.FAKE_IMAGES_DIR, 'sd-15')
real_dir = os.path.join(config.REAL_IMAGES_DIR, 'sd-15')

fake_files = list_image_files(fake_dir)[:1]
real_files = list_image_files(real_dir)[:1]

if fake_files:
    print('Extracting features for a FAKE image...')
    fake_img = Image.open(fake_files[0]).convert('RGB')
    fake_components = inverter.extract_components(fake_img)
    plot_inversion_features(
        fake_components['original'],
        fake_components['noise_map'],
        fake_components['reconstruction'],
        caption=fake_components['caption'],
        title='FAKE Image (SD-1.5)',
        save_path=os.path.join(config.RESULTS_DIR, 'demo_fake_inversion.png'),
    )
    plt.show()

if real_files:
    print('Extracting features for a REAL image...')
    real_img = Image.open(real_files[0]).convert('RGB')
    real_components = inverter.extract_components(real_img)
    plot_inversion_features(
        real_components['original'],
        real_components['noise_map'],
        real_components['reconstruction'],
        caption=real_components['caption'],
        title='REAL Image',
        save_path=os.path.join(config.RESULTS_DIR, 'demo_real_inversion.png'),
    )
    plt.show()

In [ ]:
# Side-by-side comparison
if fake_files and real_files:
    from evaluation.visualize import plot_real_vs_fake_comparison
    plot_real_vs_fake_comparison(
        real_components, fake_components,
        save_path=os.path.join(config.RESULTS_DIR, 'demo_comparison.png'),
    )
    plt.show()

In [ ]:
# Full batch feature extraction
# This is the most time-consuming step (~30s per image on T4 GPU)
from inversion.extract_features import extract_all_features

# Free previous model memory
inverter.unload_models()
del inverter
import torch
torch.cuda.empty_cache()

# Extract features (supports resume if session disconnects)
extract_all_features(subset=True)

In [ ]:
# Verify extracted features
import glob

for split in ['train', 'val', 'test', 'eval']:
    feat_dir = os.path.join(config.FEATURES_DIR, split)
    if os.path.exists(feat_dir):
        n = len(glob.glob(os.path.join(feat_dir, '*.pt')))
        print(f'{split:10s}: {n} feature files')
        
        # Check a sample
        if n > 0:
            sample = torch.load(glob.glob(os.path.join(feat_dir, '*.pt'))[0], weights_only=False)
            print(f'  Shape: {sample["features"].shape}')
            print(f'  Label: {sample["label"]}')
            print(f'  Model: {sample["model_name"]}')

## ✅ Feature extraction complete!

Next: Run **Notebook 03** to train the classifier.